В этом задании необходимо познакомиться с дообучением BERTlike моделей и сравнить их с GPT few/zero-shot подходом.
Начнем с загрузки датасета и выбора необходимых столбцов

In [2]:
import pandas as pd

# Загрузка данных
train_df = pd.read_csv('in_domain_train.csv')[['sentence', 'acceptable']]
test_df = pd.read_csv('in_domain_dev.csv')[['sentence', 'acceptable']]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7869, 2)
Test shape: (983, 2)


Поделим обучающие данные на трейн и валидацию

In [2]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2,
    random_state=42,
    stratify=train_df['acceptable']
)

print("Train:", train_data.shape)
print("Val:", val_data.shape)


Train: (6295, 2)
Val: (1574, 2)


Загружаем токенизатор и готовим датасет

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Выбор модели (пример: rubert-base-cased)
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["sentence"],  # <-- берём именно поле "sentence"
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized["labels"] = examples["acceptable"]  # добавляем labels
    return tokenized

# Конвертируем в Dataset
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_df)

# Токенизация
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)


f:\HT8\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 983/983 [00:00<00:00, 4305.39 examples/s]


Загружаем модель и задаем параметры обучения

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    do_eval=True,                    # Включить валидацию
    eval_accumulation_steps=500,         # Шаги между валидацией
    save_steps=500,             # Шаги между сохранением
    logging_steps=100            # Шаги между логами
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 569.17it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.

In [5]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Используем GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU не найден, используем CPU")

model.to(device)  # Переносим модель на GPU

Используем GPU: NVIDIA GeForce GTX 1660 Ti


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

Дообучаем модель

In [6]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized
)

trainer.train()


Step,Training Loss
100,0.571164
200,0.545970
300,0.543504
400,0.530717
500,0.467511
600,0.452960
700,0.444538
800,0.426007
900,0.307702
1000,0.277423


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]


TrainOutput(global_step=1182, training_loss=0.43126717194687897, metrics={'train_runtime': 564.4373, 'train_samples_per_second': 33.458, 'train_steps_per_second': 2.094, 'total_flos': 1242213070118400.0, 'train_loss': 0.43126717194687897, 'epoch': 3.0})

На отложенной выборке проверяем качество модели

In [9]:
import numpy as np

predictions = trainer.predict(test_tokenized)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import accuracy_score, f1_score
print("Accuracy:", accuracy_score(test_df['acceptable'], pred_labels))
print("F1:", f1_score(test_df['acceptable'], pred_labels))


Accuracy: 0.8036622583926755
F1: 0.8789968652037617


Теперь проверим как справится с этой задачей GPT

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

In [12]:
gpt_model_name = "ai-forever/rugpt3large_based_on_gpt2"
tokenizer_gpt = AutoTokenizer.from_pretrained(gpt_model_name)
model_gpt = AutoModelForCausalLM.from_pretrained(gpt_model_name, device_map="auto", torch_dtype=torch.float16)
generator = pipeline("text-generation", model=model_gpt, tokenizer=tokenizer_gpt, device=0)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [01:13<00:00,  4.01it/s, Materializing param=transformer.wte.weight]             
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Задаем функцию по генерации ответа. Перед этим я проверял моедль без затравки в промте, что давало несколько худшие показатели. Кроме того, был сдвиг в сторону ложно положительных ответов. Для исправления этого в zero-shot подходе были добавлены отрицательные примеры в промт.

In [ ]:
def predict_rugpt(sentence: str, k: int = 0, debug: bool = False) -> int:
    if k > 0:
        examples = train_df.sample(n=k, random_state=42)
    else:
        examples = pd.DataFrame()

    prompt = """Я пошёл вчера в магазин вчера. Это предложение грамматически верно? нет

Вчера я пошёл магазин. Это предложение грамматически верно? нет

Кто-то пришёл и сказал что будет дождь. Это предложение грамматически верно? нет

"""

    for _, ex in examples.iterrows():
        label_str = "да" if ex['acceptable'] == 1 else "нет"
        prompt += f"{ex['sentence']} Это предложение грамматически верно? {label_str}\n\n"

    # основной промпт — <== здесь меняй шаблон для экспериментов
    prompt += f"{sentence} Это предложение грамматически верно? "

    try:
        gen_output = generator(
            prompt,
            max_new_tokens=5,               # 4–6 оптимально
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.5,         # или 1.6–1.8, чтобы меньше "но", "конечно", "я и"
            top_k=20,                       # или top_k=1 для ещё большей жёсткости
            pad_token_id=tokenizer.eos_token_id,
        )[0]["generated_text"]


        continuation = gen_output[len(prompt):].strip().lower()

        # ────────────── ОТЛАДКА ──────────────
        if debug:
            print(f"Sentence: {sentence}")
            print(f"Prompt ends with: {prompt[-60:]}")
            print(f"Continuation: →{continuation}← (первые 60 символов)")
            print(f"Full generated:\n{gen_output}\n{'─'*100}\n")

        # парсинг
        text = continuation[:40].lower()
        if any(w in text for w in ["да", "yes", "правильно", "корректно"]):
            return 1
        if any(w in text for w in ["нет", "no", "неправильно", "ошибк"]):
            return 0
        return 0  # bias в сторону большинства класса в RuCoLA

    except Exception as e:
        print(f"Ошибка на предложении '{sentence}': {e}")
        return 0

Первые 20 ответов выводятся в режиме отладки для контроля корректности

In [43]:
predictions = []
true_labels = []

for idx, row in test_df.iterrows():
    pred = predict_rugpt(row["sentence"], k=0, debug=(idx < 20))
    predictions.append(pred)
    true_labels.append(row["acceptable"])

    if idx == 100:
        break
# Метрики
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

print(f"\nРезультаты на zero-shot ({len(predictions)} примеров)")
print(f"Accuracy:  {accuracy_score(true_labels, predictions):.4f}")
print(f"F1:        {f1_score(true_labels, predictions):.4f}")
print(f"MCC:       {matthews_corrcoef(true_labels, predictions):.4f}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence: Иван вчера не позвонил.
Prompt ends with: ван вчера не позвонил. Это предложение грамматически верно? 
Continuation: →- нет, это← (первые 60 символов)
Full generated:
Иван вчера не позвонил. Это предложение грамматически верно? 
- Нет, это
────────────────────────────────────────────────────────────────────────────────────────────────────



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence: У многих туристов, кто посещают Кемер весной, есть шанс застать снег на вершине горы Тахталы и даже сочетать пляжный отдых с горнолыжным.
Prompt ends with: й отдых с горнолыжным. Это предложение грамматически верно? 
Continuation: →- да! мы← (первые 60 символов)
Full generated:
У многих туристов, кто посещают Кемер весной, есть шанс застать снег на вершине горы Тахталы и даже сочетать пляжный отдых с горнолыжным. Это предложение грамматически верно? 
- Да! Мы
────────────────────────────────────────────────────────────────────────────────────────────────────



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Sentence: Но Коле не помог его иноверец.
Prompt ends with: не помог его иноверец. Это предложение грамматически верно? 
Continuation: →- нет, -← (первые 60 символов)
Full generated:
Но Коле не помог его иноверец. Это предложение грамматически верно? 
- Нет, -
────────────────────────────────────────────────────────────────────────────────────────────────────



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence: Удивительно милый, честный, добрый человек, всегда готовый откликнуться на чужое горе, на слово которого можно положиться.
Prompt ends with: рого можно положиться. Это предложение грамматически верно? 
Continuation: →- да! -← (первые 60 символов)
Full generated:
Удивительно милый, честный, добрый человек, всегда готовый откликнуться на чужое горе, на слово которого можно положиться. Это предложение грамматически верно? 
- Да! -
────────────────────────────────────────────────────────────────────────────────────────────────────



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end


Результаты на zero-shot (101 примеров)
Accuracy:  0.5743
F1:        0.6815
MCC:       0.0747


В конце получились довольно неплохие показатели, проверим улучшатся ли они с ростом количества примеров в промте

In [54]:
predictions = []
true = []

for i, row in test_df.iterrows():
    pred = predict_rugpt(row['sentence'], k=1)
    predictions.append(pred)
    true.append(row['acceptable'])

from sklearn.metrics import accuracy_score, f1_score

print("\nРезультаты для 1-shot:")
print(f"Accuracy: {accuracy_score(true, predictions):.4f}")
print(f"F1:       {f1_score(true, predictions):.4f}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end


Результаты для 1-shot:
Accuracy: 0.6124
F1:       0.7319


In [58]:
predictions = []
true = []

for i, row in test_df.iterrows():
    pred = predict_rugpt(row['sentence'], k=2)
    predictions.append(pred)
    true.append(row['acceptable'])

from sklearn.metrics import accuracy_score, f1_score

print("\nРезультаты для 2-shot:")
print(f"Accuracy: {accuracy_score(true, predictions):.4f}")
print(f"F1:       {f1_score(true, predictions):.4f}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end


Результаты для 2-shot:
Accuracy: 0.6124
F1:       0.7319


In [56]:
predictions = []
true = []

for i, row in test_df.iterrows():
    pred = predict_rugpt(row['sentence'], k=3)
    predictions.append(pred)
    true.append(row['acceptable'])

from sklearn.metrics import accuracy_score, f1_score

print("\nРезультаты для 3-shot:")
print(f"Accuracy: {accuracy_score(true, predictions):.4f}")
print(f"F1:       {f1_score(true, predictions):.4f}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end


Результаты для 3-shot:
Accuracy: 0.6124
F1:       0.7319


In [57]:
predictions = []
true = []

for i, row in test_df.iterrows():
    pred = predict_rugpt(row['sentence'], k=4)
    predictions.append(pred)
    true.append(row['acceptable'])

from sklearn.metrics import accuracy_score, f1_score

print("\nРезультаты для 4-shot:")
print(f"Accuracy: {accuracy_score(true, predictions):.4f}")
print(f"F1:       {f1_score(true, predictions):.4f}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end


Результаты для 4-shot:
Accuracy: 0.6124
F1:       0.7319


К сожалению рост количества примеров в промте не дал ожидаемого прироста в качестве. Вероятно предел качества достигается при суммарно 4 примерах с учетов изначально прописанных в промте.

In [3]:
results = [
    {"Тест": "DeepPavlov/rubert-base-cased",      "Accuracy": 0.8037, "F1": 0.8789},
    {"Тест": "rugpt3large_based_on_gpt2 zero-shot", "Accuracy": 0.5743, "F1": 0.6815},
    {"Тест": "1-shot",      "Accuracy": 0.6124, "F1": 0.7319},
    {"Тест": "2-shot",      "Accuracy": 0.6124, "F1": 0.7319},
    {"Тест": "3-shot",      "Accuracy": 0.6124, "F1": 0.7319},
    {"Тест": "4-shot",      "Accuracy": 0.6124, "F1": 0.7319},
]

# Создаём DataFrame
results = pd.DataFrame(results)

# Выводим таблицу
print(results.to_string(index=False))


                               Тест  Accuracy     F1
       DeepPavlov/rubert-base-cased    0.8037 0.8789
rugpt3large_based_on_gpt2 zero-shot    0.5743 0.6815
                             1-shot    0.6124 0.7319
                             2-shot    0.6124 0.7319
                             3-shot    0.6124 0.7319
                             4-shot    0.6124 0.7319


На мой взгляд использование BERTlike моделей для данной задачи наиболее оправдано в виду их легковесности и легкой обучаемости. Дообучение т5 модели не проводилось по причине ограниченности ресурсов компьютера.